# 02. Thu thập và tích hợp dữ liệu (Data Collection)

Luồng dữ liệu, mỗi mũi tên đều là SQL:

```
food.parquet (8 GB, Hugging Face)
   │  DuckDB: SELECT ... WHERE ... ORDER BY md5(mã || seed) LIMIT 30000      (src/fetch_from_dump.py)
   ▼
data/raw/raw_products_dump_<thời điểm>.jsonl.gz   (JSON gốc + thời điểm lấy + nguồn)
   │  INSERT ... ON CONFLICT DO NOTHING                                      (src/load_raw_to_postgres.py)
   ▼
PostgreSQL: bảng raw_products (cột payload kiểu JSONB) = RAW LAYER
   │  SELECT payload->>'code', payload->'ingredients_tags', CASE ... END ... (sql/extract_products.sql)
   ▼
data/processed/products.parquet   (bảng phẳng 30.000 dòng = Raw Dataset cho EDA)
```

In [1]:
# Chạy từ thư mục gốc dự án để đọc được data/, sql/ và import src/
import os, sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

## 4.1. Lấy mẫu từ bản dump bằng SQL (DuckDB)

DuckDB chạy SQL **thẳng trên tệp Parquet**, chỉ đọc những cột được nêu tên, nên không phải nạp 8 GB vào RAM. Lấy mẫu chia hai pha:
- **Pha 1** chọn 30.000 mã vạch đủ điều kiện. `ORDER BY md5(mã || seed)` là một cách trộn ngẫu nhiên nhưng **tái lập được**: cùng seed 42 luôn ra cùng mẫu.
- **Pha 2** đọc đủ cột cho các mã đã chọn. Bản dump có vài mã bị lặp, nên `QUALIFY row_number() ... = 1` giữ bản sửa gần nhất.

In [2]:
from src import fetch_from_dump as fd

SRC = fd.SOURCE
print(fd.PICK_SQL.format(src=SRC, eligible=fd.ELIGIBLE))
print(fd.ROWS_SQL.format(columns=", ".join(fd.COLUMNS), src=SRC, eligible=fd.ELIGIBLE))


CREATE OR REPLACE TEMP TABLE picked AS
SELECT code FROM (SELECT DISTINCT code FROM read_parquet('data/external/food.parquet') WHERE lower(nutriscore_grade) IN ('a', 'b', 'c', 'd', 'e') AND len(ingredients_tags) > 0 AND code IS NOT NULL AND coalesce(obsolete, false) = false)
ORDER BY md5(code || ?) LIMIT ?


SELECT code, product_name, brands_tags, countries_tags, categories_tags, food_groups_tags, ingredients_tags, ingredients_text, ingredients_n, ingredients_from_palm_oil_n, ingredients_analysis_tags, additives_tags, additives_n, allergens_tags, labels_tags, nova_group, nutriscore_grade, nutriscore_score, nutriments, lang, last_modified_t FROM read_parquet('data/external/food.parquet')
WHERE code IN (SELECT code FROM picked) AND lower(nutriscore_grade) IN ('a', 'b', 'c', 'd', 'e') AND len(ingredients_tags) > 0 AND code IS NOT NULL AND coalesce(obsolete, false) = false
QUALIFY row_number() OVER (PARTITION BY code ORDER BY last_modified_t DESC NULLS LAST) = 1
ORDER BY md5(code || ?)



In [3]:
# Có bao nhiêu sản phẩm đủ điều kiện để chọn mẫu?
import duckdb

duckdb.sql(f"SELECT count(DISTINCT code) AS so_ma_du_dieu_kien FROM read_parquet('{SRC}') WHERE {fd.ELIGIBLE}").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,so_ma_du_dieu_kien
0,943455


Lệnh lấy mẫu thật (chạy một lần trong cmd, mất vài phút nên không chạy lại trong notebook):

```bat
python -m src.fetch_from_dump --n 30000 --seed 42
```

Mỗi dòng của tệp kết quả gồm `_metadata` (thời điểm lấy, nguồn, tham số) và `raw_payload` (JSON gốc của sản phẩm). Đây là yêu cầu "raw layer có timestamp và nguồn".

In [4]:
import gzip, json

RAW_FILE = "data/raw/raw_products_dump_20261017_174142.jsonl.gz"   # tệp đã nạp vào PostgreSQL
with gzip.open(RAW_FILE, "rt", encoding="utf-8") as f:
    first = json.loads(next(f))

print(json.dumps(first["_metadata"], indent=2, ensure_ascii=False))
payload = first["raw_payload"]
{k: payload[k] for k in ["code", "product_name", "nutriscore_grade", "nova_group", "pnns_groups_2"]} | {
    "ingredients_tags (8 thẻ đầu)": payload["ingredients_tags"][:8]}

{
  "retrieved_at": "2026-10-17T17:41:46.516320+00:00",
  "source_endpoint": "data/external/food.parquet",
  "request_url": null,
  "http_status": null,
  "page": null,
  "filters": {
    "strategy": "natural",
    "n": 30000,
    "per_grade": null,
    "country": null,
    "seed": "42"
  }
}


{'code': '0051500825631',
 'product_name': 'Peanut Butter & Strawberry Jam Sandwich',
 'nutriscore_grade': 'e',
 'nova_group': 4,
 'pnns_groups_2': 'one dish meals',
 'ingredients_tags (8 thẻ đầu)': ['en:bread',
  'en:barley-malt-flour',
  'en:cereal',
  'en:flour',
  'en:cereal-flour',
  'en:barley',
  'en:barley-flour',
  'en:e375']}

## 4.2. Nạp vào PostgreSQL (raw layer)

Bảng `raw_products` giữ nguyên JSON gốc trong cột `payload` (kiểu JSONB), kèm thời điểm lấy và tên tệp nguồn. Ràng buộc `UNIQUE (code, source_file)` + `ON CONFLICT DO NOTHING` làm cho việc nạp lại cùng tệp không sinh dòng trùng.

```bat
psql -U postgres -d food_facts_db -f sql\init_raw_db.sql
python -m src.load_raw_to_postgres data\raw\raw_products_dump_20261017_174142.jsonl.gz
```

Mật khẩu CSDL đọc từ tệp `.env` (không đưa lên Git); `src/db.py` lo phần kết nối.

In [5]:
print(open("sql/init_raw_db.sql", encoding="utf-8").read())

CREATE TABLE IF NOT EXISTS raw_products (
    id               BIGSERIAL PRIMARY KEY,
    code             TEXT,
    retrieved_at     TIMESTAMPTZ,
    source_endpoint  TEXT,
    request_url      TEXT,
    http_status      INTEGER,
    page             INTEGER,
    source_file      TEXT NOT NULL,
    payload          JSONB NOT NULL,
    UNIQUE (code, source_file)
);

CREATE INDEX IF NOT EXISTS idx_raw_products_code ON raw_products (code);



In [6]:
from src.db import query, load_queries

query('''
    SELECT source_file, count(*) AS so_ban_ghi, min(retrieved_at) AS thoi_diem_lay
    FROM raw_products
    GROUP BY source_file
''')

,source_file,so_ban_ghi,thoi_diem_lay
0,raw_products_dump_20261017_174142.jsonl.gz,30000,2026-10-18 00:41:46.516320+07:00


## 4.3. Trích xuất bằng SQL: JSONB → bảng phẳng

Thay vì viết vòng lặp Python để "làm phẳng" JSON, một câu `SELECT` làm hết:

| Cú pháp PostgreSQL | Ý nghĩa |
|---|---|
| `payload->'ingredients_tags'` | lấy giá trị JSON (mảng giữ nguyên, Python nhận về list) |
| `payload->>'code'` | lấy giá trị dạng chữ |
| `NULLIF(trim(x), '')` | chuỗi rỗng → NULL (thiếu) |
| `CASE WHEN jsonb_typeof(x) = 'array' THEN x END` | chỉ nhận đúng kiểu, sai kiểu → NULL |
| `CASE WHEN x ~ '^[1-4]$' ...` | kiểm tra bằng biểu thức chính quy trước khi đổi kiểu số |
| `row_number() OVER (ORDER BY id) - 1` | `row_id` cố định giữa các lần chạy |

Câu SQL **không lọc dòng nào**: lọc và loại trùng là việc của bước chuẩn bị dữ liệu (notebook 04), để còn đếm được mỗi bước loại bao nhiêu.

In [7]:
SQL = load_queries("sql/extract_products.sql")["products"]
print(SQL)

SELECT
    row_number() OVER (ORDER BY id) - 1                                  AS row_id,   -- thứ tự cố định theo id
    NULLIF(trim(payload->>'code'), '')                                   AS code,     -- giữ dạng chuỗi (số 0 đầu)
    NULLIF(trim(payload->>'product_name'), '')                           AS product_name,
    -- Danh sách thẻ: chỉ nhận mảng JSON, còn lại để NULL
    CASE WHEN jsonb_typeof(payload->'brands_tags') = 'array' THEN payload->'brands_tags' END            AS brands_tags,
    CASE WHEN jsonb_typeof(payload->'countries_tags') = 'array' THEN payload->'countries_tags' END      AS countries_tags,
    CASE WHEN jsonb_typeof(payload->'categories_tags') = 'array' THEN payload->'categories_tags' END    AS categories_tags,
    CASE WHEN jsonb_typeof(payload->'ingredients_tags') = 'array' THEN payload->'ingredients_tags' END  AS ingredients_tags,
    CASE WHEN jsonb_typeof(payload->'ingredients_analysis_tags') = 'array'
         THEN payload->'ingredients_analysis_tags' 

In [8]:
products = query(SQL)
print("Kích thước:", products.shape)
products.head(3)

Kích thước: (30000, 29)


,row_id,code,product_name,brands_tags,countries_tags,categories_tags,ingredients_tags,ingredients_analysis_tags,additives_tags,allergens_tags,...,last_modified,nutriscore_grade,nutriscore_score,energy_kcal_100g,fat_100g,saturated_fat_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g
0,0,0051500825631,Peanut Butter & Strawberry Jam Sandwich,[xx:Smuckers Uncrustables],[en:united-states],[en:meals],"[en:bread, en:barley-malt-flour, en:cereal, en:flour, en:cereal-flour, en:barley, en:b...","[en:may-contain-palm-oil, en:vegan-status-unknown, en:vegetarian-status-unknown]","[en:e202, en:e300, en:e330, en:e440, en:e471, en:e930]","[en:gluten, en:peanuts, en:soybeans]",...,2026-06-08 22:37:36+07:00,e,19.0,362.06897,15.517241,3.448276,17.241379,3.448276,10.344828,1.633621
1,1,7610198074344,Caffè Latte Zero,[xx:volg],[en:switzerland],"[en:plant-based-foods-and-beverages, en:beverages, en:plant-based-foods, en:coffees, e...","[en:nahrwerte-valeurs-nutritives-koffein, en:cafeine-energiewert, en:valeur-energetiqu...","[en:palm-oil-content-unknown, en:vegan-status-unknown, en:vegetarian-status-unknown]",[],[],...,2026-04-02 05:17:56+07:00,b,0.0,39.00000,1.300000,0.800000,3.600000,NaN,2.800000,0.100000
2,2,3222476292353,Taboulé oriental,[xx:Casino],[en:france],"[en:meals, en:fresh-foods, en:savory-semolina-dishes, en:fresh-meals, en:tabbouleh, en...","[en:durum-wheat, en:cereal, en:wheat, en:tomato, en:vegetable, en:fruit-vegetable, en:...","[en:palm-oil-free, en:vegan, en:vegetarian]",[],"[en:gluten, en:mustard]",...,2026-03-20 10:24:53+07:00,c,6.0,174.00000,6.000000,0.700000,3.500000,2.000000,4.000000,0.900000


In [9]:
products.to_parquet("data/processed/products.parquet", index=False)

# Sản phẩm ví dụ dùng xuyên suốt các notebook
EXAMPLE = "3457221000505"
products[products["code"] == EXAMPLE].T.head(22)

,206
row_id,206
code,3457221000505
product_name,Caramel au beurre salé
brands_tags,[xx:creperie de coataudon]
countries_tags,[en:france]
categories_tags,"[en:breakfasts, en:spreads, en:sweet-spreads, fr:pates-a-tartiner, en:caramel-spreads,..."
ingredients_tags,"[en:sugar, en:added-sugar, en:disaccharide, en:single-cream, en:dairy, en:cream, en:gl..."
ingredients_analysis_tags,"[en:palm-oil-free, en:non-vegan, en:vegetarian]"
additives_tags,[]
allergens_tags,[en:milk]


**Kết quả (Output: Raw Dataset):** `data/processed/products.parquet`, 30.000 sản phẩm × 29 cột, gồm:
- cột đặc trưng (thành phần, phụ gia, danh mục, NOVA, nhãn, quốc gia);
- nhãn `nutriscore_grade` (y);
- điểm và bảng dinh dưỡng **chỉ để EDA** (notebook 04 tách chúng ra trước khi làm đặc trưng).

Mã vạch giữ kiểu chuỗi: 10.963 mã bắt đầu bằng số 0 sẽ bị mất nếu đổi sang số.